In [ ]:
import os
import sys
import cv2
import torch
import random
import numpy as np

from pathlib import Path
from PIL import Image, ImageFilter
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel
import torchvision.transforms.functional as TF
import torch.nn.functional as F

import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/home/cavadalab/Documents/scsv/fungitastic2026_2")
DATASET_ROOT = Path("/data0/sebastian.cavada/datasets/FungiTastic")
OUTPUT_ROOT = PROJECT_ROOT / "data_processed"
MODEL_NAME = os.environ.get("MODEL_NAME", "facebook/dinov3-vit7b16-pretrain-lvd1689m")
OUTPUT_NAME = os.environ.get("OUTPUT_NAME", "")
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "64"))
NUM_WORKERS = int(os.environ.get("NUM_WORKERS", "8"))
SHARD_SIZE = int(os.environ.get("SHARD_SIZE", "512"))
SEED = 0
DTYPE = torch.bfloat16

sys.path.append(str(PROJECT_ROOT / "FungiTastic"))
from dataset.mask_fungi import MaskFungiTastic
from dataset.utils.mask_vis import get_image_shape, resize_mask_to_image

SPLIT = os.environ.get("SPLIT", "train")
DEFAULT_BATCH_SIZE = 8
MODEL_LOAD_DTYPE = DTYPE
FEATURE_DTYPE = DTYPE
DATA_SUBSET = "all"
DATASET_SIZE = os.environ.get("DATASET_SIZE", "300")
IMAGE_SIZE = 224 if DATASET_SIZE == "300" else 448
TASK = "closed"
SEG_TASK = "binary"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = DEVICE == "cuda"
BACKGROUND_TYPES = [
    # "crop",
    # "crop_black",
    # "masked_black",
    # "masked_blurred", # not needed for now what I am doing is a different analysis now
    "normal",
]
BACKGROUND = "normal"
MIN_BOX_AREA = 2500

/home/cavadalab/Documents/scsv/fungitastic2026_2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/cavadalab/Documents/scsv/fungitastic2026_2/.venv/lib/python3.12/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [ ]:
def collate_batch(batch):
    images = [item[0] for item in batch]
    # from /home/cavadalab/Documents/scsv/fungitastic2026/FungiTastic/dataset/mask_fungi.py
    masks = [torch.from_numpy(resize_mask_to_image(item[1], get_image_shape(item[0]))).unsqueeze(0) for item in batch]    
    labels = [item[2] for item in batch]
    file_paths = [item[3] for item in batch]
    return images, masks, labels, file_paths

dataset = MaskFungiTastic(
        root=str(DATASET_ROOT),
        split=SPLIT,
        size=DATASET_SIZE,
        task=TASK,
        data_subset=DATA_SUBSET,
        transform=None,
        seg_task=SEG_TASK,
        workers=8,
    )

dataloader_kwargs = {
        "batch_size": 1,
        "shuffle": False,
        "num_workers": NUM_WORKERS,
        "collate_fn": collate_batch,
    }

dataloader = DataLoader(dataset, **dataloader_kwargs)


Generated color mapping for 9 unique labels: ['cap', 'fruiting_body', 'gills', 'microscopic', 'pores', 'ring', 'stem', 'teeth', 'unknown underside']


In [ ]:
# utils
PATH_SEGMENT = "/home/cavadalab/Documents/scsv/fungitastic2026_2/data_processed/sam3_yolo/all/train/720/FungiTastic/train/720p"

def visualize_masks_gt_sam(image, gt_mask, sam_mask, alpha=0.5):
    """
    Overlays GT (Green) and SAM (Red) on the original image.
    alpha: transparency of the masks (0 to 1)
    """    
    # Convert PIL Image to NumPy array
    if not isinstance(image, np.ndarray):
        image = np.array(image)
        
    # 1. Ensure image is in RGB and normalized to [0, 1] for plotting
    if image.max() > 1.0:
        image = image.astype(np.float32) / 255.0
        
    # 2. Create the overlay image
    overlay = image.copy()    
    
    # 3. Apply Ground Truth in Green [R, G, B]
    # Where gt_mask > 0, we blend the green channel
    overlay[gt_mask > 0] = overlay[gt_mask > 0] * (1 - alpha) + np.array([0, 1, 0]) * alpha
    
    # 4. Apply SAM Prediction in Red [R, G, B]
    # Where sam_mask > 0, we blend the red channel
    overlay[sam_mask > 0] = overlay[sam_mask > 0] * (1 - alpha) + np.array([1, 0, 0]) * alpha

    # 5. Plotting
    plt.figure(figsize=(15, 7))
    
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(image)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title("Overlay (GT=Green, SAM=Red)")
    plt.imshow(overlay)
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

def polygon_to_mask(polygons, img_width, img_height):
    """
    Converts a list of polygons into a single binary mask.
    segments: List of polygons, where each polygon is a list of (x, y) tuples.
    """
    # 1. Initialize an empty black mask
    mask = np.zeros((img_height, img_width), dtype=np.uint8)
    
    # 2. Prepare the list of scaled polygons for OpenCV
    all_polygons_scaled = []
    
    for polygon in polygons:
        # Scale the normalized (0-1) coordinates to actual pixel values
        pixel_coords = np.array([[x * img_width, y * img_height] for x, y in polygon], dtype=np.int32)
        all_polygons_scaled.append(pixel_coords)
    
    # 3. Fill all polygons with white (255)
    # cv2.fillPoly can take a list of arrays directly
    if all_polygons_scaled:
        cv2.fillPoly(mask, all_polygons_scaled, 255)
    
    return mask

def read_segments(file_name):
    segment_path = os.path.join(PATH_SEGMENT, file_name.replace(".jpg", ".png"))
    segments = []
    class_ids = []

    if os.path.exists(segment_path):
        with open(segment_path, "r") as f:
            lines = f.readlines()

            for line in lines:                
                parts = line.strip().split(" ")
                class_id = int(parts[0])
                points = parts[1:]
                polygon = [(float(points[i]), float(points[i + 1])) for i in range(0, len(points), 2)]
                segments.append(polygon)
                class_ids.append(class_id)
        
        return segments, class_ids
        # return torch.from_numpy(np.array(segment)).unsqueeze(0)  # Add channel dimension
    else:
        print(f"Segment not found for {file_name}")
        return None

In [68]:
# Limit to the first 5 batches
limit = 13777
visualize_masks_gt_sam_flag = False
avg_iou = 0

for i, (images, masks, labels, file_paths) in enumerate(tqdm(dataloader, desc="Processing", unit="batch")):    
    if i < limit:
        image_name = Path(file_paths[0]).stem
        # print(f"Processing batch with image: {image_name}")
        segments, class_ids = read_segments(image_name + ".txt")
        
        if segments is not None:
            # print(f"Segments found for {image_name}: {len(segments)} segments")
            
            mask_sam = polygon_to_mask(segments, images[0].size[0], images[0].size[1])
            mask_sam_tensor = torch.from_numpy(mask_sam).unsqueeze(0)                    
            intersection = np.logical_and(masks[0][0].numpy() > 0, mask_sam_tensor[0].numpy() > 0).sum()
            union = np.logical_or(masks[0][0].numpy() > 0, mask_sam_tensor[0].numpy() > 0).sum()
            iou = intersection / union if union > 0 else 0
            avg_iou += iou

            if visualize_masks_gt_sam_flag:
                visualize_masks_gt_sam(images[0], masks[0][0], mask_sam_tensor[0])
        else:
            print(f"No segments found for {image_name}")
            avg_iou += 0

       

    else:
        print(f"Average IoU over {limit} batches: {avg_iou / limit:.4f}")
        # Optional: break if you only want to test the first few
        break 
        # pass

Processing: 100%|██████████| 13777/13777 [00:24<00:00, 555.21batch/s]


In [69]:
print(f"Average IoU over {limit} batches: {avg_iou / limit:.4f}")

Average IoU over 13777 batches: 0.8737
